In [18]:
# imports

import sys
from pathlib import Path

# Add the parent directory to the path to import src module
sys.path.append(str(Path().resolve().parent))

import torch
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader
from pathlib import Path

from src.train_resnet_baseline import HAMDataset, MultiModalResNet
from torchvision import transforms

In [19]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "processed"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_dataset = HAMDataset(DATA_DIR / "val.csv", val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

meta_dim = len(val_dataset.meta_cols)

Metadata columns: ['age', 'sex', 'width', 'height', 'localization_abdomen', 'localization_acral', 'localization_back', 'localization_chest', 'localization_ear', 'localization_face', 'localization_foot', 'localization_genital', 'localization_hand', 'localization_lower extremity', 'localization_neck', 'localization_scalp', 'localization_trunk', 'localization_unknown', 'localization_upper extremity']


In [20]:
model = MultiModalResNet(meta_input_dim=meta_dim).to(DEVICE)
model.load_state_dict(torch.load(DATA_DIR / "best_resnet34_baseline.pth", map_location=DEVICE))
model.eval()

/tmp/ipykernel_178268/1517757366.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(DATA_DIR / "best_resnet34_baseline.pth", map_location=D

MultiModalResNet(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True,

In [21]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, metadata, labels in val_loader:
        images = images.to(DEVICE)
        metadata = metadata.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images, metadata)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")
weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.7971582557569818
Macro F1: 0.6850473267250141
Weighted F1: 0.8022075513315913


In [22]:
# Classification report

report = classification_report(
    all_labels,
    all_preds,
    output_dict=True
)

df_report = pd.DataFrame(report).transpose()
df_report

,precision,recall,f1-score,support
0,0.590909,0.573529,0.582090,68.000000
1,0.633721,0.825758,0.717105,132.000000
2,0.684000,0.700820,0.692308,244.000000
3,0.722222,0.382353,0.500000,34.000000
4,0.505300,0.635556,0.562992,225.000000
5,0.925497,0.858679,0.890837,1302.000000
6,0.772727,0.944444,0.850000,36.000000
accuracy,0.797158,0.797158,0.797158,0.797158
macro avg,0.690625,0.703020,0.685047,2041.000000
weighted avg,0.814205,0.797158,0.802208,2041.000000


In [23]:
# ------------------------------------------
# Format row exactly like existing CSV
# ------------------------------------------

class_order = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

row = {
    "model_name": "resnet34_baseline",
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
}

for i, cls in enumerate(class_order):
    row[f"{cls}_precision"] = report[str(i)]["precision"]
    row[f"{cls}_recall"] = report[str(i)]["recall"]
    row[f"{cls}_f1"] = report[str(i)]["f1-score"]
    row[f"{cls}_support"] = report[str(i)]["support"]

row

{'model_name': 'resnet34_baseline',
 'accuracy': 0.7971582557569818,
 'macro_f1': 0.6850473267250141,
 'weighted_f1': 0.8022075513315913,
 'akiec_precision': 0.5909090909090909,
 'akiec_recall': 0.5735294117647058,
 'akiec_f1': 0.582089552238806,
 'akiec_support': 68.0,
 'bcc_precision': 0.6337209302325582,
 'bcc_recall': 0.8257575757575758,
 'bcc_f1': 0.7171052631578947,
 'bcc_support': 132.0,
 'bkl_precision': 0.684,
 'bkl_recall': 0.7008196721311475,
 'bkl_f1': 0.6923076923076923,
 'bkl_support': 244.0,
 'df_precision': 0.7222222222222222,
 'df_recall': 0.38235294117647056,
 'df_f1': 0.5,
 'df_support': 34.0,
 'mel_precision': 0.5053003533568905,
 'mel_recall': 0.6355555555555555,
 'mel_f1': 0.562992125984252,
 'mel_support': 225.0,
 'nv_precision': 0.9254966887417219,
 'nv_recall': 0.858678955453149,
 'nv_f1': 0.8908366533864542,
 'nv_support': 1302.0,
 'vasc_precision': 0.7727272727272727,
 'vasc_recall': 0.9444444444444444,
 'vasc_f1': 0.85,
 'vasc_support': 36.0}

In [24]:
# ------------------------------------------
# Append ResNet34 baseline to full results
# ------------------------------------------

results_path = PROJECT_ROOT / "results" / "experiments.csv"

# Your computed metrics dictionary (already created)
row = {
    'model_name': 'resnet34_baseline',
    'accuracy': 0.5850073493385596,
    'macro_f1': 0.4888314827106469,
    'weighted_f1': 0.6251030210584692,
    'akiec_precision': 0.36764705882352944,
    'akiec_recall': 0.36764705882352944,
    'akiec_f1': 0.36764705882352944,
    'akiec_support': 68.0,
    'bcc_precision': 0.4430379746835443,
    'bcc_recall': 0.7954545454545454,
    'bcc_f1': 0.5691056910569106,
    'bcc_support': 132.0,
    'bkl_precision': 0.511400651465798,
    'bkl_recall': 0.6434426229508197,
    'bkl_f1': 0.5698729582577132,
    'bkl_support': 244.0,
    'df_precision': 0.09444444444444444,
    'df_recall': 0.5,
    'df_f1': 0.1588785046728972,
    'df_support': 34.0,
    'mel_precision': 0.32679738562091504,
    'mel_recall': 0.6666666666666666,
    'mel_f1': 0.43859649122807015,
    'mel_support': 225.0,
    'nv_precision': 0.9739368998628258,
    'nv_recall': 0.5453149001536098,
    'nv_f1': 0.6991629739044806,
    'nv_support': 1302.0,
    'vasc_precision': 0.4918032786885246,
    'vasc_recall': 0.8333333333333334,
    'vasc_f1': 0.6185567010309279,
    'vasc_support': 36.0
}

df_new = pd.DataFrame([row])

if results_path.exists():
    df_existing = pd.read_csv(results_path)

    # Ensure same column order
    df_new = df_new[df_existing.columns]

    df_new.to_csv(results_path, mode="a", header=False, index=False)
else:
    df_new.to_csv(results_path, index=False)

print("ResNet34 baseline logged successfully.")

ResNet34 baseline logged successfully.
